# Zadanie 1, Część II – Word Embeddings (3 pkt) · PolEmo 2.0 · Word2Vec

Notebook odpowiada punktom z instrukcji (Word Embeddings, 3 pkt):

| Pkt zadania | Punkty | Sekcja w notebooku |
|---|---|---|
| 1. Osadzenia słów w anotowanym korpusie (co najmniej 2 modele) | 1 | **Sekcja 1** |
| 2. t-SNE z odniesieniem do etykiet, **wykres interaktywny** (po najechaniu: słowo i anotacja) | 0,5 | **Sekcja 2** |
| 3. Listy k najbliższych słów i ich porównanie dla 2 modeli | 1 | **Sekcja 3** |
| 4. Krótka dyskusja | 0,5 | **Sekcja 4** |

Modele: **Word2Vec IPI PAN** (ogólny, NKJP + Wikipedia, 300 tys. form) i **Word2Vec PolEmo** (wytrenowany w gensim
na zdaniach PolEmo). Kod: `src/word_embeddings.py`, `src/word_tsne.py`, `src/word_neighbours.py`; w notebooku jest
tylko uruchomienie i komentarz. Notebook czyta wyniki zapisane w `results/word_embeddings/`, więc działa bez
pobierania modeli (`RECOMPUTE = True` przelicza pkt 1 i 3, wtedy modele muszą być w `models/`).

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.io as pio
from IPython.display import display

import config
from src import word_embeddings as we, word_tsne as wt, word_neighbours as wn

pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)
# Wykresy interaktywne w notebooku: „plotly_mimetype” rysują JupyterLab, Notebook 7 i VS Code,
# „notebook_connected” – klasyczny Notebook i eksport do HTML (plotly.js z CDN, potrzebny internet).
# Bez tego ustawienia Plotly poza notebookiem otwiera wykres w przeglądarce zamiast pod komórką.
pio.renderers.default = "plotly_mimetype+notebook_connected"
OUT = config.WORD_EMB_DIR

## Konfiguracja

In [ ]:
MODELS    = ["w2v", "w2v_polemo"]   # Word2Vec IPI PAN i Word2Vec wytrenowany na PolEmo
SUBSETS   = ["all", "annotated"]    # wszystkie słowa z wektorem / tylko słowa z anotacją różną od O
RECOMPUTE = False                   # True = przelicz pkt 1 i 3 (wymaga modeli w models/, zob. README_WordEmbeddings.md)

## 1. Osadzenia słów w anotowanym korpusie [1 pkt]

Słowo = token z 200 zaanotowanych zdań (iteracje 1–2) zawierający literę, zapisany małymi literami. Dla każdego
słowa zapisujemy etykiety nadane przez anotatorów, **etykietę słowa** (najczęstsza etykieta różna od O, jeśli co
najmniej połowa anotacji słowa jest różna od O) i rozkład etykiet zdań, w których wystąpiło. Wektory: 100 wymiarów.

In [ ]:
if RECOMPUTE:
    we.run(MODELS)
vocab = we.read_vocab()
cov = json.loads((OUT / "coverage.json").read_text(encoding="utf-8"))
print(f"Słownik zaanotowanego korpusu: {len(vocab)} słów, {int(vocab['freq'].sum())} wystąpień")
display(pd.DataFrame({
    m: {"model": cov["models"][m], "słownik modelu": cov["vocab_size"][m],
        "słowa z wektorem": f"{c['types']['in_vocab']}/{c['types']['n']} ({c['types']['share']:.1%})",
        "wystąpienia z wektorem": f"{c['tokens']['share']:.1%}",
        "słowa zaanotowane z wektorem": f"{c['annotated_types']['share']:.1%}"}
    for m, c in cov["coverage"].items()}).T)
cols = ["word", "freq", "word_label", "annot_share", "share_POZ", "share_NEG", "share_NEU", "share_AMB"]
display(vocab[vocab["word_label"] != "O"][cols].head(10))

## 2. t-SNE z odniesieniem do etykiet anotacji – wykresy interaktywne [0,5 pkt]

Każdy punkt to słowo. **Po najechaniu na punkt** widać: słowo (i jego pisownię w korpusie), **etykietę słowa**,
liczbę anotacji każdej etykiety, liczbę wystąpień, dominującą etykietę zdań (z rozkładem procentowym), częstość
w całym PolEmo i rangę słowa w modelu.

* **Lista rozwijana** nad wykresem zmienia kolorowanie: *etykieta słowa* (WYR_POZ / WYR_NEG / WZM / OSL / ODW / O)
  albo *kontekst zdań* (POZ / NEG / NEU / AMB / mieszany).
* Przycisk **„podpisy”** pokazuje napisy przy słowach z etykietą różną od O występujących co najmniej 2 razy.
* **Kliknięcie w legendę** ukrywa lub pokazuje klasę (np. szare tło O). Wielkość punktu rośnie z liczbą wystąpień.
* t-SNE: odległość kosinusowa, inicjalizacja PCA, perplexity 30, seed 42 (ta sama metoda co dla zdań).

Kolory: zielony = pozytywne, pomarańczowy = negatywne, fioletowy = modyfikatory (WZM ▲, OSL ▼, ODW ✕) i AMB;
słowa bez etykiety i zdania NEU są szarym tłem.

In [ ]:
def show_tsne(model, subset):
    """Liczy t-SNE wektorów słów i wyświetla wykres interaktywny pod komórką."""
    points, X = wt.load_points(model, subset)
    xy = wt.tsne_coords(X)
    title = f"t-SNE słów – {wt.MODEL_TITLES[model]} – {wt.SUBSETS[subset]} (n = {len(points)})"
    wt.make_word_figure(points, xy, title, rank_col=f"{model}_rank").show()

### 2.1 Word2Vec IPI PAN – wszystkie słowa z wektorem

In [ ]:
show_tsne("w2v", "all")

### 2.2 Word2Vec IPI PAN – tylko słowa z anotacją (t-SNE liczone osobno)

In [ ]:
show_tsne("w2v", "annotated")

### 2.3 Word2Vec PolEmo – wszystkie słowa z wektorem

In [ ]:
show_tsne("w2v_polemo", "all")

### 2.4 Word2Vec PolEmo – tylko słowa z anotacją

In [ ]:
show_tsne("w2v_polemo", "annotated")

### 2.5 Czy etykiety się grupują? (sprawdzenie w oryginalnej przestrzeni 100 wymiarów)

t-SNE zniekształca odległości, więc grupowanie sprawdzamy też liczbowo: **czystość k-NN** = odsetek 10 najbliższych
sąsiadów z tą samą etykietą, **losowo** = to samo po przetasowaniu etykiet (1000 razy), **sylwetka** (bliska 0 = brak
skupisk), **sonda** = trafność prostego klasyfikatora odczytującego etykietę z wektora.

In [ ]:
sep = json.loads((OUT / "label_separation.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {"model": wt.MODEL_TITLES.get(m, m), "zestaw": name, "n": r["n"], "czystość k-NN": round(r["purity"], 3),
     "losowo": round(r["random_mean"], 3), "p": r["p_value"], "sylwetka": round(r["silhouette"], 3),
     "sonda (trafność)": round(r["probe"]["balanced_accuracy"], 3) if "probe" in r else None}
    for m, sets in sep.items() for name, r in sets.items()]))

**Obserwacje.** Słowa WYR_POZ i WYR_NEG są na wykresach wymieszane. Słowa o tej samej etykiecie są sobie
nieco bliższe niż przy losowych etykietach, ale nie tworzą skupisk (sylwetka ≈ 0); skupiają się głównie słowa
pozytywne. Antonimy leżą blisko, bo Word2Vec uczy się z kontekstu, a słowa o przeciwnym wydźwięku występują
w tych samych kontekstach. Wyraźniej grupują się słowa według funkcji: modyfikatory (*bardzo, bardziej, trochę,
całkiem*) tworzą zwarte skupisko. Prosty klasyfikator odczytuje jednak wydźwięk z wektorów IPI PAN lepiej niż
losowo – informacja jest w wektorach, choć nie w sąsiedztwie słów.

## 3. k najbliższych słów i porównanie modeli [1 pkt]

Dla słów z etykietą różną od O, znanych obu modelom, 10 najbliższych słów (kosinus, tylko słowa z liter, z całego
słownika modelu). Pełne listy: `results/word_embeddings/neighbours_<model>.csv`, opis: `knn_summary.md`.

In [ ]:
if RECOMPUTE:
    wn.run(MODELS)
knn = json.loads((OUT / "knn_summary.json").read_text(encoding="utf-8"))
pair = next(iter(knn["pairs"]))
print(f"Zgodność list ({pair}); średnie po słowach zapytania:")
display(pd.DataFrame(knn["pairs"][pair]).T[["n", "overlap", "overlap_shared", "rho2"]].round(3)
        .rename(columns={"overlap": "wspólne@10", "overlap_shared": "wspólne@10 (wspólny słownik)",
                         "rho2": "ρ drugiego rzędu"}))
lists = {m: pd.read_csv(OUT / f"neighbours_{m}.csv", encoding="utf-8-sig", keep_default_na=False) for m in MODELS}
top5 = pd.DataFrame({wt.MODEL_TITLES[m]: lists[m].sort_values("rank").groupby("query")["neighbour"]
                     .apply(lambda s: ", ".join(s.head(5))) for m in MODELS})
display(top5.loc[[w for w in knn["showcase"] if w in top5.index]])

## 4. Dyskusja [0,5 pkt]

Pełna dyskusja (odpowiedzi na cztery pytania pomocnicze) jest w sprawozdaniu: `report/sections/we_dyskusja.tex`
(rozdział „Część II: Word Embeddings”). W skrócie:

* **Pokrycie i znaczenie ogólne:** model IPI PAN zna ok. 95% słów korpusu i trafnie oddaje znaczenie ogólne
  (*lekarz* → *chirurg, internista*), ale nie wydźwięk; model PolEmo łapie znaczenie z recenzji
  (*miła* → *profesjonalna, kompetentna*), kosztem szumu dla rzadkich słów.
* **Grupowanie według anotacji:** słabe – antonimy leżą blisko (cos(*dobry*, *zły*) ≈ 0,73), więc słowa
  pozytywne i negatywne się mieszają (sekcja 2).
* **Różnice list:** modele mają tylko ok. 10% wspólnych sąsiadów; model ogólny daje relacje słownikowe, model
  z recenzji – skojarzenia tematyczne (sekcja 3).
* **Ulepszenia:** dotrenowanie modelu ogólnego na recenzjach, wiedza o wydźwięku (retrofitting), fastText
  (n-gramy znakowe), lematyzacja, negacja jako jednostka, modele kontekstowe.